# RL upper-bound improver for unknotting numbers (stronger invariant version)

This notebook keeps the **same self-improving pipeline** as `upper_bound_unknotting_v6_local.ipynb`,
but replaces Jones-polynomial identification with a **knot Floer homology key** (a stronger invariant).

Expected local layout:

```text
repo-root/
├─ upper_bound_unknotting_v7_strong_invariant.ipynb
├─ requirements.txt
├─ README.md
├─ data/
│  └─ unknotting.xlsx
├─ models/
│  └─ best_model.zip                 # optional
├─ training_data/                    # optional
│  ├─ hard_unknots.csv
│  ├─ very_hard_unknots.csv
│  └─ random_diagrams.csv
└─ outputs/
```

The notebook does the following:

1. loads `data/unknotting.xlsx`
2. fills missing Floer-invariant keys from PD presentations when feasible
3. finds knots whose unknotting number is given by a range such as `[2,3]`
4. inflates the diagram
5. flips one crossing at a time
6. runs the RL unknotter / reducer
7. computes the Floer-invariant key of the reduced knot
8. identifies matches in the local workbook database, allowing mirrors
9. takes the **maximum** matching upper bound
10. improves the original upper bound by setting it to `[old_lower, 1 + matched_upper_bound]`
11. saves the workbook after each processed knot

As in the paper, identification is only trusted when the reduced diagram lies in the configured
database crossing range.


In [ ]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt


In [ ]:

import os, re, json, ast, math, random, csv, glob, time
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Local repository paths (no Google Drive needed)
from pathlib import Path

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "data").exists() or (cand / "models").exists() or (cand / "training_data").exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
TRAINING_DIR = REPO_ROOT / "training_data"
OUT_DIR = REPO_ROOT / "outputs"

for folder in [DATA_DIR, MODELS_DIR, TRAINING_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_DIR / "unknotting.xlsx"
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Missing workbook: {XLSX_PATH}\n"
        "Place your database at repo-root/data/unknotting.xlsx"
    )

BASE = REPO_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("Workbook :", XLSX_PATH)
print("MODELS   :", MODELS_DIR)
print("TRAINING :", TRAINING_DIR)
print("OUTPUTS  :", OUT_DIR)


In [ ]:
# ----------------------------
# Configuration
# ----------------------------
# Which unresolved knots to process:
#   PROCESS_MODE = "all"              -> all range-valued unknotting entries, i.e. all [a,b] with a != b
#   PROCESS_MODE = "first_n"          -> first N range-valued entries
#   PROCESS_MODE = "slice"            -> entries [START_INDEX:END_INDEX] among range-valued entries
#   PROCESS_MODE = "bounds_eq"        -> only entries with unknotting number exactly [TARGET_LOWER, TARGET_UPPER]
#   PROCESS_MODE = "bounds_eq_slice"  -> first filter to [TARGET_LOWER, TARGET_UPPER], then take [START_INDEX:END_INDEX]
#   PROCESS_MODE = "bounds_neq"       -> explicitly all [a,b] with a != b (same target set as "all")
#   PROCESS_MODE = "bounds_neq_slice" -> explicitly all [a,b] with a != b, then take [START_INDEX:END_INDEX]
PROCESS_MODE = "slice"

TARGET_LOWER = 1
TARGET_UPPER = 2

START_INDEX = 0          # 0-based within the filtered target list
END_INDEX = 10           # exclusive; e.g. 100 means "first 100"
FIRST_N = 100            # used only when PROCESS_MODE == "first_n"

NUM_VARIANTS_PER_KNOT = 12
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20

UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500

TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# Strong invariant settings (knot Floer homology key)
USE_KNOT_FLOER = True
MAX_CROSSINGS_FOR_HFK = 25   # skip rows above this when precomputing lookup keys

# Saving / output
SAVE_AFTER_EACH_TARGET = True
SAVE_EVERY_KNOT = True          # robust alias used in the main loop
WRITE_UPDATED_COPY = False
MAKE_TIMESTAMPED_BACKUP = False
# No backup file and no secondary updated copy: only overwrite unknotting.xlsx

# Only trust database identification when the reduced PD lies in this range.
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

# Optional safety valve for very long runs:
MAX_FLIPS_PER_VARIANT = None   # set to an integer to cap crossing flips per inflated variant

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Training data sources from the original notebook / paper pipeline
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Optional local extras: only used if present
LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]


In [ ]:
# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
_int_pat = re.compile(r'-?\d+')

def pick_first_existing(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_pd_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, list):
        return [[int(y) for y in q] for q in x]
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list) and all(isinstance(q, (list, tuple)) and len(q) == 4 for q in obj):
            return [[int(y) for y in q] for q in obj]
    except Exception:
        pass
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', s)
    if items:
        out = []
        for it in items:
            nums = [int(z.strip()) for z in it.split(',')]
            if len(nums) != 4:
                return None
            out.append(nums)
        return out
    return None

def parse_unknotting_entry(x):
    """
    Returns dict with:
      kind: 'missing' | 'exact' | 'range' | 'other'
      lower, upper: ints or None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}
    if isinstance(x, (int, np.integer)):
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}
    if isinstance(x, float) and float(x).is_integer():
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}

    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) == 2:
            a, b = int(obj[0]), int(obj[1])
            if a == b:
                return {"kind": "exact", "lower": a, "upper": b, "raw": x}
            return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}
    except Exception:
        pass

    nums = [int(z) for z in _int_pat.findall(s)]
    if len(nums) == 1:
        return {"kind": "exact", "lower": nums[0], "upper": nums[0], "raw": x}
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        if a == b:
            return {"kind": "exact", "lower": a, "upper": b, "raw": x}
        return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}

    return {"kind": "other", "lower": None, "upper": None, "raw": x}

def format_unknotting(lower, upper):
    if lower is None and upper is None:
        return None
    if lower is None or upper is None:
        return None
    return str([int(lower), int(upper)])

def normalize_invariant_key_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, dict):
        return json.dumps(x, sort_keys=True, separators=(",", ":"))
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("{") and s.endswith("}"):
        try:
            obj = json.loads(s)
            return json.dumps(obj, sort_keys=True, separators=(",", ":"))
        except Exception:
            return s
    return s


In [ ]:
# ----------------------------
# Load workbook and identify columns
# ----------------------------
df = pd.read_excel(XLSX_PATH)

knot_col   = pick_first_existing(df, ["knot_id", "name", "knot", "id"])
pd_col     = pick_first_existing(df, ["pd_presentation", "pd_notation", "pd", "pd_code"])
u_col      = pick_first_existing(df, ["unknotting_number", "unknotting", "u"])
strong_col = pick_first_existing(df, ["strong_invariant_key", "floer_invariant_key", "alex_sig_det_key"])

print("Columns:")
print("  knot_col  :", knot_col)
print("  pd_col    :", pd_col)
print("  u_col     :", u_col)
print("  strong_col:", strong_col)

if knot_col is None or pd_col is None or u_col is None:
    raise ValueError("Could not identify the required knot / PD / unknotting-number columns.")

if strong_col is None:
    strong_col = "strong_invariant_key"
    df[strong_col] = None
    print(f"Created missing strong-invariant column: {strong_col}")

display(df.head())


In [ ]:
# ----------------------------
# Strong invariant code (knot Floer homology)
# ----------------------------

def _as_int_or_none(x):
    if x is None:
        return None
    try:
        return int(x)
    except Exception:
        return None

def normalize_hfk_dict(hfk_obj):
    if not isinstance(hfk_obj, dict):
        return None

    out = {
        "L_space_knot": None if hfk_obj.get("L_space_knot") is None else bool(hfk_obj.get("L_space_knot")),
        "fibered": None if hfk_obj.get("fibered") is None else bool(hfk_obj.get("fibered")),
        "modulus": _as_int_or_none(hfk_obj.get("modulus")),
        "seifert_genus": _as_int_or_none(hfk_obj.get("seifert_genus")),
        "tau": _as_int_or_none(hfk_obj.get("tau")),
        "nu": _as_int_or_none(hfk_obj.get("nu")),
        "epsilon": _as_int_or_none(hfk_obj.get("epsilon")),
        "total_rank": _as_int_or_none(hfk_obj.get("total_rank")),
    }

    ranks = hfk_obj.get("ranks", {})
    norm_ranks = []
    if isinstance(ranks, dict):
        for k, v in ranks.items():
            try:
                a, m = k
                r = int(v)
            except Exception:
                continue
            norm_ranks.append([int(a), int(m), int(r)])
    norm_ranks.sort(key=lambda t: (t[0], t[1], t[2]))
    out["ranks"] = norm_ranks

    if out["total_rank"] is None and norm_ranks:
        out["total_rank"] = int(sum(t[2] for t in norm_ranks))

    return out

def hfk_key_from_link(L):
    hfk = L.knot_floer_homology()
    norm = normalize_hfk_dict(hfk)
    if norm is None:
        raise ValueError("knot_floer_homology returned non-dict output")
    return json.dumps(norm, sort_keys=True, separators=(",", ":"))

HFK_KEY_CACHE = {}

def hfk_keys_from_pd(pd_list, include_mirror=False):
    if not pd_list:
        return None, None, "Empty PD"

    pd_canon = json.dumps(pd_list, separators=(",", ":"))
    cache_key = ("mirror" if include_mirror else "direct", pd_canon)
    if cache_key in HFK_KEY_CACHE:
        k, km = HFK_KEY_CACHE[cache_key]
        return k, km, None

    try:
        L = Link(pd_list)
        k = hfk_key_from_link(L)
        km = None
        if include_mirror:
            km = hfk_key_from_link(L.mirror())

        HFK_KEY_CACHE[cache_key] = (k, km)
        # also seed direct cache for the same PD
        if ("direct", pd_canon) not in HFK_KEY_CACHE:
            HFK_KEY_CACHE[("direct", pd_canon)] = (k, None)

        return k, km, None
    except Exception as e:
        return None, None, repr(e)


In [ ]:
# ----------------------------
# Fill missing Floer-invariant keys in the workbook itself when possible
# ----------------------------
missing_before = 0
filled = 0
errors = []
skipped_too_large = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling Floer invariant keys"):
    key = normalize_invariant_key_cell(row.get(strong_col))
    if key is not None:
        df.at[idx, strong_col] = key
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > MAX_CROSSINGS_FOR_HFK:
        skipped_too_large += 1
        continue

    new_key, _, err = hfk_keys_from_pd(pd_list, include_mirror=False)
    if new_key is not None:
        df.at[idx, strong_col] = new_key
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Floer-invariant entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Skipped due to crossing cap:", skipped_too_large)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])


In [ ]:
# ----------------------------
# Build Floer-invariant lookup from the workbook
# Requested rule:
#   if multiple knots share the same invariant key,
#   take the MAX over all matching upper bounds
# Mirrors are included by indexing both the knot key and its mirror key.
# ----------------------------
lookup = defaultdict(list)
missing_key_rows = 0
skipped_too_large = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building Floer invariant lookup"):
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    key = normalize_invariant_key_cell(row.get(strong_col))
    mirror_key = None

    if key is None:
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is None:
            missing_key_rows += 1
            continue

        if len(pd_list) > MAX_CROSSINGS_FOR_HFK:
            skipped_too_large += 1
            continue

        key, mirror_key, err = hfk_keys_from_pd(pd_list, include_mirror=True)
        if key is None:
            missing_key_rows += 1
            continue
        df.at[idx, strong_col] = key
    else:
        # existing key available; compute mirror key from PD when possible
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None and len(pd_list) <= MAX_CROSSINGS_FOR_HFK:
            _, mirror_key, _ = hfk_keys_from_pd(pd_list, include_mirror=True)

    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }
    lookup[key].append({**rec, "mirror": False})
    if mirror_key is not None and mirror_key != key:
        lookup[mirror_key].append({**rec, "mirror": True})

print("Lookup keys:", len(lookup))
print("Rows skipped due to crossing cap:", skipped_too_large)
print("Rows skipped due to missing/uncomputable invariant key:", missing_key_rows)


In [ ]:
# ----------------------------
# Training / RL utilities, adapted from the uploaded notebook
# ----------------------------
import io
import gzip

# Candidate local model files
MODEL_PATH_CANDIDATES = [
    MODELS_DIR / "best_model.zip",
    MODELS_DIR / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Optional local training files
LOCAL_EXTRA_FILES = [
    TRAINING_DIR / "hard_unknots.csv",
    TRAINING_DIR / "very_hard_unknots.csv",
    TRAINING_DIR / "random_diagrams.csv",
    TRAINING_DIR / "random_diagrams.txt",
    DATA_DIR / "hard_unknots.csv",
    DATA_DIR / "very_hard_unknots.csv",
]

def parse_link_strict(s: str):
    s = s.strip()
    try:
        obj = ast.literal_eval(s)
    except Exception as e:
        raise ValueError(f"Could not parse PD/DT string: {s[:80]}") from e
    if isinstance(obj, list):
        return Link(obj)
    raise ValueError("Parsed training example is not a list-like PD/DT object.")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)
            if crossings(L) == c0:
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    raw = []
    for _, row in df.iterrows():
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None:
            raw.append(json.dumps(pd_list))
            if max_keep is not None and len(raw) >= max_keep:
                break
    return raw

@dataclass
class EnvCfg:
    max_steps: int = UNKNOTTER_MAX_STEPS
    step_penalty: float = 0.05
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    w_delta: float = 1.0
    w_uphill: float = 0.5
    w_potential: float = 0.02

class SphKnotEnv(gym.Env):
    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)
        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        self.L = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        return min(3, self.num_actions - 1)

    def _obs(self):
        c = crossings(self.L)
        try:
            comps = len(self.L.link_components)
        except Exception:
            comps = 1
        tmp = Link(self.L.PD_code())
        try:
            reduced = tmp.simplify(mode="basic")
        except TypeError:
            reduced = tmp.simplify()
        can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
        recently_dropped = float(self._last_drop)
        after_backtrack = 1.0 if self._after_backtrack else 0.0
        return np.array([c, comps, can_reduce, recently_dropped, after_backtrack, self._steps], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        pd_str = self.rng.choice(self.pd_lines)
        self.L = parse_link_strict(pd_str)
        return self._obs(), {}

    def step(self, action):
        mode = int(action[0])
        cap = int(action[1]) if len(np.array(action).shape) > 0 else 0
        mode = self._map_blocked_mode(mode)
        self._steps += 1
        old_c = crossings(self.L)
        reward = -self.cfg.step_penalty
        terminated = False
        truncated = False
        info = {"crossings": old_c, "mode": mode, "cap": cap}

        try:
            if mode == 0:
                try:
                    self.L.simplify(mode="basic")
                except TypeError:
                    self.L.simplify()
            elif mode == 1:
                self.L.backtrack(steps=max(1, cap))
                self._after_backtrack = True
            elif mode == 2:
                self.L, _ = riii_shuffle_only_link(self.L, max(1, cap))
            else:
                self.L.simplify(mode="level")
        except Exception:
            self._blocked[mode] = True

        new_c = crossings(self.L)
        delta = old_c - new_c
        self._last_drop = max(0, delta)
        reward += self.cfg.w_delta * delta
        if delta < 0:
            reward += self.cfg.w_uphill * delta

        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            terminated = True

        if self._steps >= self.cfg.max_steps:
            truncated = True

        info["crossings"] = new_c
        return self._obs(), reward, terminated, truncated, info

def load_training_pd_lines():
    raw = []

    for path in LOCAL_EXTRA_FILES:
        if path.exists():
            try:
                extra = read_first_col_local(str(path), has_header=True)
                raw += extra
                print(f"Loaded local extra {path.name}: {len(extra)}")
            except Exception as e:
                print(f"Could not load {path.name}: {e}")

    if not raw:
        raw = workbook_pd_lines()
        print(f"Falling back to workbook PD data: {len(raw)} examples")

    pd_lines = clean_pd_lines(raw, max_keep=None)
    random.Random(SEED).shuffle(pd_lines)
    if not pd_lines:
        raise RuntimeError(
            "No valid PD/DT strings available for training. "
            "Add local CSV/TXT files under training_data/ or ensure unknotting.xlsx contains PD data."
        )
    return pd_lines

def make_single_env(pd_list, cfg: EnvCfg):
    pd_str = json.dumps(pd_list)
    pd_lines_single = [pd_str]
    def _make():
        return SphKnotEnv(pd_lines_single, cfg)
    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg, episodes: int = 3, return_best_pd: bool = False):
    vec_env = make_single_env(pd_list, cfg)
    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(q) for q in pd_list]

    for _ in range(episodes):
        obs = vec_env.reset()
        best_crossings_ep = best_crossings_global
        best_pd_ep = best_pd_global

        for _step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)
            info = infos[0]
            cr = info.get("crossings", None)
            try:
                cur_pd = [list(q) for q in vec_env.envs[0].L.PD_code()]
            except Exception:
                cur_pd = None

            if cr is not None and cr < best_crossings_ep and cur_pd is not None:
                best_crossings_ep = cr
                best_pd_ep = cur_pd

            if cr == 0:
                success = True
                if cur_pd is not None:
                    best_crossings_ep = 0
                    best_pd_ep = cur_pd
                break
            if dones[0]:
                break

        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep
        if success:
            break

    vec_env.close()
    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global


In [ ]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True)

# Robust fallback in case the config cell was edited or executed out of order.
if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = [
        BASE / "best_model.zip",
        BASE / "ppo_knot_rl_spherogram_continued.zip",
        OUT_DIR / "best_model.zip",
    ]

best_model_path = None
for path in MODEL_PATH_CANDIDATES:
    if Path(path).exists():
        best_model_path = Path(path)
        break

if best_model_path is not None:
    model = PPO.load(str(best_model_path), device="auto")
    print("Loaded existing model:", best_model_path)
else:
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            "No trained model found in MODEL_PATH_CANDIDATES, and TRAIN_IF_MODEL_MISSING=False."
        )
    print("No existing model found. Training a fresh one.")
    print("Searched these locations:")
    for p in MODEL_PATH_CANDIDATES:
        print("  -", p)

    pd_lines_train = load_training_pd_lines()
    vec_env = DummyVecEnv([lambda: SphKnotEnv(pd_lines_train, cfg)])
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=SEED,
        verbose=1,
    )
    model.learn(total_timesteps=TRAIN_STEPS_IF_NEEDED, progress_bar=True)
    best_model_path = OUT_DIR / "best_model.zip"
    model.save(str(best_model_path))
    vec_env.close()
    print("Saved trained model to:", best_model_path)

In [ ]:
# ----------------------------
# Inflation and flip helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_inflated_variants(pd_list,
                               num_variants=10,
                               backtrack_steps_min=6,
                               backtrack_steps_max=8,
                               riii_steps_max=20):
    variants = []
    L0 = Link(pd_list)

    for _ in range(num_variants):
        pd0 = L0.PD_code()
        if isinstance(pd0, list):
            L = Link(pd0)
        else:
            L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

        steps = random.randint(backtrack_steps_min, backtrack_steps_max)
        try:
            L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
        except Exception:
            pass
        try:
            L, _ = riii_shuffle_only_link(L, min(riii_steps_max, steps))
        except Exception:
            pass

        try:
            pd_new = [list(q) for q in L.PD_code()]
            variants.append(pd_new)
        except Exception:
            variants.append([list(q) for q in pd_list])

    if not variants:
        variants = [[list(q) for q in pd_list]]
    return variants

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append((i, flipped))
    return out

def match_hfk_key_to_database(inv_key):
    if inv_key is None:
        return []
    return lookup.get(inv_key, [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)


In [ ]:
# ----------------------------
# Choose target rows
# ----------------------------
all_range_targets = []
for idx, row in df.iterrows():
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["kind"] == "range":
        all_range_targets.append({
            "row_index": int(idx),
            "knot": row.get(knot_col),
            "lower": int(uk["lower"]),
            "upper": int(uk["upper"]),
        })

print("All range-valued unknotting rows (all [a,b] with a != b):", len(all_range_targets))

bounds_targets = [
    t for t in all_range_targets
    if t["lower"] == TARGET_LOWER and t["upper"] == TARGET_UPPER
]
print(f"Rows with unknotting range [{TARGET_LOWER},{TARGET_UPPER}]:", len(bounds_targets))

neq_targets = list(all_range_targets)
print("Rows with non-exact unknotting range [a,b], a != b:", len(neq_targets))

if PROCESS_MODE == "all":
    targets = list(all_range_targets)
elif PROCESS_MODE == "first_n":
    targets = list(all_range_targets[:FIRST_N])
elif PROCESS_MODE == "slice":
    targets = list(all_range_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_eq":
    targets = list(bounds_targets)
elif PROCESS_MODE == "bounds_eq_slice":
    targets = list(bounds_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_neq":
    targets = list(neq_targets)
elif PROCESS_MODE == "bounds_neq_slice":
    targets = list(neq_targets[START_INDEX:END_INDEX])
else:
    raise ValueError(f"Unknown PROCESS_MODE: {PROCESS_MODE}")

print("Selected targets:", len(targets))
if len(targets) > 0:
    preview = pd.DataFrame(targets[:10])
    display(preview)
else:
    print("No targets selected.")


In [ ]:
# ----------------------------
# Main improvement loop
# ----------------------------
# Robust defaults in case cells were run out of order
if "SAVE_EVERY_KNOT" not in globals():
    SAVE_EVERY_KNOT = True
if "MIN_DATABASE_CROSSINGS" not in globals():
    MIN_DATABASE_CROSSINGS = 1
if "MAX_DATABASE_CROSSINGS" not in globals():
    MAX_DATABASE_CROSSINGS = 13

results_all = []
timestamp = time.strftime("%Y%m%d-%H%M%S")

for tnum, target in enumerate(targets, start=1):
    idx = target["row_index"]
    knot_name = target["knot"]
    current_lower = target["lower"]
    current_upper = target["upper"]

    print()
    print("="*80)
    print(f"[{tnum}/{len(targets)}] Processing:", knot_name, " current range:", [current_lower, current_upper])

    original_pd = parse_pd_cell(df.at[idx, pd_col])
    if original_pd is None:
        print("  Skipping: bad or missing PD.")
        results_all.append({
            "row_index": idx,
            "knot": knot_name,
            "status": "bad_pd",
        })
        continue

    base_key = normalize_invariant_key_cell(df.at[idx, strong_col])
    if base_key is None and len(original_pd) <= MAX_CROSSINGS_FOR_HFK:
        new_key, _, err = hfk_keys_from_pd(original_pd, include_mirror=False)
        if new_key is not None:
            df.at[idx, strong_col] = new_key

    variants = generate_inflated_variants(
        original_pd,
        num_variants=NUM_VARIANTS_PER_KNOT,
        backtrack_steps_min=BACKTRACK_STEPS_MIN,
        backtrack_steps_max=BACKTRACK_STEPS_MAX,
        riii_steps_max=RIII_STEPS_MAX,
    )

    best_new_upper = current_upper
    best_evidence = None
    tried = 0
    matched = 0

    for vnum, variant_pd in enumerate(variants, start=1):
        flip_variants = generate_single_flip_variants(variant_pd)

        for flip_index, flipped_pd in flip_variants:
            tried += 1
            success, min_crossings_found, best_pd = run_unknotter_on_pd(
                flipped_pd,
                model,
                cfg,
                episodes=UNKNOTTER_EPISODES_PER_FLIP,
                return_best_pd=True,
            )

            if best_pd is None:
                continue

            reduced_crossings = len(best_pd)

            # Only trust matching inside the configured database crossing range.
            if not (MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS):
                continue

            if reduced_crossings > MAX_CROSSINGS_FOR_HFK:
                continue

            inv_key, _, err = hfk_keys_from_pd(best_pd, include_mirror=False)
            if inv_key is None:
                continue

            matches = match_hfk_key_to_database(inv_key)
            if not matches:
                continue

            matched += 1
            matched_upper = best_upper_bound_from_matches(matches)
            if matched_upper is None:
                continue

            candidate_upper = matched_upper + 1

            if candidate_upper < best_new_upper:
                best_new_upper = candidate_upper
                best_evidence = {
                    "variant_index": vnum,
                    "flip_index": int(flip_index),
                    "matched_upper": int(matched_upper),
                    "candidate_upper": int(candidate_upper),
                    "rl_success": bool(success),
                    "min_crossings_found": int(min_crossings_found),
                    "best_pd_crossings": reduced_crossings,
                    "matched_knots": sorted(
                        {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in matches},
                        key=lambda x: (x[0], x[1], x[2])
                    ),
                    "best_pd": best_pd,
                    "hfk_invariant_key": inv_key,
                }
                print(f"  Improved via variant {vnum}, flip {flip_index}: upper {current_upper} -> {best_new_upper}")

    improved = best_new_upper < current_upper
    if improved:
        df.at[idx, u_col] = format_unknotting(current_lower, best_new_upper)

        if SAVE_EVERY_KNOT:
            df.to_excel(XLSX_PATH, index=False)
            print("  Saved updated workbook to:", XLSX_PATH)
    else:
        print("  No improvement found.")

    result = {
        "row_index": idx,
        "knot": knot_name,
        "old_lower": current_lower,
        "old_upper": current_upper,
        "new_upper": best_new_upper,
        "improved": improved,
        "tried_flip_runs": tried,
        "matched_flip_runs": matched,
        "evidence": best_evidence,
    }
    results_all.append(result)

    jsonl_path = OUT_DIR / "upper_bound_pass_results_strong_invariant.jsonl"
    with open(jsonl_path, "a") as f:
        f.write(json.dumps(result))
        f.write(chr(10))

print()
print("Done.")


This notebook is the stronger-invariant analogue of `upper_bound_unknotting_v6_local.ipynb`.
